importaciones, carga de datos y deficion de variables

In [ ]:
#librerias
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

#1. CARGA DE DATOS Y DEFINICIÓN DE CONSTANTES

datos = np.load("datos_606.npz")
X = datos['X']
y = datos['y']
t = datos['t']

N = 256
K = 30
ITERACIONES = 4000
LEARNING_RATE = 0.02

X0 = X[y == 0]
X1 = X[y == 1]
X2 = X[y == 2]

-----------------------FUNCIONES MATEMATICAS Y DE RED--------------------------

Funciones de configuracion (Setup)

In [ ]:
def construir_matriz_fourier(N=256, K=30):
    n = jnp.arange(N)
    filas = []

    for k in range(1, K + 1):
        filas.append(jnp.cos(2 * jnp.pi * k * n / N))
        filas.append(jnp.sin(2 * jnp.pi * k * n / N))

    return (2.0 / N) * jnp.stack(filas)

def coeficientes_fourier_directos(x, K=30):
    N = x.shape[0]
    n = jnp.arange(N)
    coeficientes = []

    for k in range(1, K + 1):
        coef_cos = (2.0 / N) * jnp.sum(x * jnp.cos(2 * jnp.pi * k * n / N))
        coef_sin = (2.0 / N) * jnp.sum(x * jnp.sin(2 * jnp.pi * k * n / N))
        coeficientes.append(coef_cos)
        coeficientes.append(coef_sin)

    return jnp.array(coeficientes)

def inicializar_fourier(key, N=256, K=30):
    W1 = construir_matriz_fourier(N, K)
    W2 = 0.01 * jax.random.normal(key, shape=(N, 2 * K))
    return {"W1": W1, "W2": W2}

def inicializar_aleatoria(key, N=256, K=30):
    key_w1, key_w2 = jax.random.split(key)
    W1 = 0.01 * jax.random.normal(key_w1, shape=(2 * K, N))
    W2 = 0.01 * jax.random.normal(key_w2, shape=(N, 2 * K))
    return {"W1": W1, "W2": W2}


Funciones del modelo y de entrenamiento.

In [ ]:

def red(params, x):
    W1 = params["W1"]
    W2 = params["W2"]
    return W2 @ jnp.tanh(W1 @ x)

red_batch = jax.vmap(red, in_axes=(None, 0))

def perdida(params, X):
    X_hat = red_batch(params, X)
    return jnp.mean((X_hat - X) ** 2)

@jax.jit
def paso_entrenamiento(params, X, learning_rate):
    valor_perdida, gradientes = jax.value_and_grad(perdida)(params, X)
    nuevos_params = jax.tree_util.tree_map(
        lambda parametro, gradiente: parametro - learning_rate * gradiente,
        params,
        gradientes
    )
    return nuevos_params, valor_perdida

def entrenar(params_iniciales, X, y, iteraciones=4000, learning_rate=0.02):
    params = params_iniciales
    X2_local = X[y == 2]

    historial_total = []
    historial_regimen2 = []

    for _ in range(iteraciones):
        params, valor_perdida = paso_entrenamiento(params, X, learning_rate)
        valor_regimen2 = perdida(params, X2_local)

        historial_total.append(float(valor_perdida))
        historial_regimen2.append(float(valor_regimen2))

    return params, np.array(historial_total), np.array(historial_regimen2)


Funciones de evaluacion y analisis.

In [ ]:
def perdida_por_regimen(params, X, y):
    resultados = {}
    for regimen in [0, 1, 2]:
        X_regimen = X[y == regimen]
        resultados[regimen] = float(perdida(params, X_regimen))
    return resultados

def energia_acumulada_coeficientes(x, W_fourier):
    coeficientes = W_fourier @ x
    energia = coeficientes ** 2

    indices = jnp.argsort(energia)[::-1]
    energia_ordenada = energia[indices]
    energia_total = jnp.sum(energia_ordenada)

    acumulada = jnp.where(
        energia_total > 0,
        jnp.cumsum(energia_ordenada) / energia_total,
        jnp.zeros_like(energia_ordenada)
    )

    return coeficientes, indices, acumulada

def coeficientes_para_porcentaje(x, W_fourier, porcentaje=0.85):
    _, _, acumulada = energia_acumulada_coeficientes(x, W_fourier)
    cantidad = jnp.argmax(acumulada >= porcentaje) + 1
    return int(cantidad)